# Developer Career & Technology Trends — EDA

This notebook explores the Stack Overflow Developer Survey 2022 dataset before building the Streamlit app.

**Questions**
1. Which programming languages are most commonly used?
2. Which technologies do respondents want to learn?
3. Which countries have the most survey respondents?
4. How is professional coding experience associated with reported compensation?
5. How does employment composition vary by education level?

> Important: this is one survey year. A single cross-sectional dataset cannot establish year-over-year technology growth. We use the difference between current usage and learning interest as a *demand/interest gap*, not as proven growth.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
df = pd.read_csv("data/survey_cleaned.csv", low_memory=False)

print("Shape:", df.shape)
display(df.head())


## 1. Data quality checks

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True)
}).sort_values("missing_percent", ascending=False)

display(quality.head(20))
print("Exact duplicate rows:", df.duplicated().sum())


## 2. Most popular programming languages

In [ ]:
def split_answers(series):
    return (series.dropna().astype(str).str.split(";").explode()
            .str.strip().replace("", pd.NA).dropna())

language_use = split_answers(df["LanguageHaveWorkedWith"]).value_counts().head(15)
display(language_use.rename_axis("Language").reset_index(name="Respondents"))

language_use.sort_values().plot(kind="barh", figsize=(9, 6), title="Top 15 languages currently used")
plt.xlabel("Selections")
plt.tight_layout()
plt.show()


## 3. Technologies developers want to learn

In [ ]:
language_want = split_answers(df["LanguageWantToWorkWith"]).value_counts().head(15)
display(language_want.rename_axis("Language").reset_index(name="Respondents"))

language_want.sort_values().plot(kind="barh", figsize=(9, 6), title="Top 15 languages developers want to learn")
plt.xlabel("Selections")
plt.tight_layout()
plt.show()


## 4. Country distribution

In [ ]:
country_counts = df["Country"].value_counts().head(15)
display(country_counts.rename_axis("Country").reset_index(name="Responses"))

country_counts.sort_values().plot(kind="barh", figsize=(9, 6), title="Top 15 countries by survey responses")
plt.xlabel("Survey responses")
plt.tight_layout()
plt.show()


## 5. Experience and salary

In [ ]:
salary = df[["YearsCodePro", "ConvertedCompYearly"]].copy()
salary["ExperienceYears"] = salary["YearsCodePro"].replace({
    "Less than 1 year": "0.5",
    "More than 50 years": "51"
})
salary["ExperienceYears"] = pd.to_numeric(salary["ExperienceYears"], errors="coerce")
salary["SalaryUSD"] = pd.to_numeric(salary["ConvertedCompYearly"], errors="coerce")
salary = salary.dropna(subset=["ExperienceYears", "SalaryUSD"])
salary = salary[(salary["SalaryUSD"] > 0) & (salary["SalaryUSD"] <= salary["SalaryUSD"].quantile(.99))]

salary["ExperienceBand"] = pd.cut(
    salary["ExperienceYears"],
    bins=[-0.01, 1, 3, 5, 10, 15, 20, 30, 60],
    labels=["<1", "1-3", "3-5", "5-10", "10-15", "15-20", "20-30", "30+"]
)
salary_summary = salary.groupby("ExperienceBand", observed=False)["SalaryUSD"].agg(
    respondents="count", median_salary="median", mean_salary="mean"
).round(0)
display(salary_summary)

salary_summary["median_salary"].dropna().plot(
    marker="o", figsize=(9, 5), title="Median reported annual compensation by professional experience"
)
plt.ylabel("Median annual compensation (USD)")
plt.xlabel("Years of professional coding experience")
plt.tight_layout()
plt.show()


## 6. Education and employment

In [ ]:
education_employment = pd.crosstab(
    df["EdLevel"], df["Employment"], normalize="index"
) * 100
display(education_employment.round(1))

education_employment.plot(kind="bar", stacked=True, figsize=(13, 6), colormap="tab20")
plt.ylabel("Share within education group (%)")
plt.xlabel("Education level")
plt.title("Employment composition by education level")
plt.xticks(rotation=35, ha="right")
plt.legend(title="Employment", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 7. Interest gap (not year-over-year growth)

In [ ]:
used = split_answers(df["LanguageHaveWorkedWith"]).value_counts()
wanted = split_answers(df["LanguageWantToWorkWith"]).value_counts()
interest_gap = pd.concat([used.rename("CurrentlyUsed"), wanted.rename("WantToLearn")], axis=1).fillna(0)
interest_gap["Gap"] = interest_gap["WantToLearn"] - interest_gap["CurrentlyUsed"]
display(interest_gap.sort_values("Gap", ascending=False).head(15))

print("Interpretation: a positive gap means more respondents selected the technology as something they want to learn than selected it as something they currently use.")
print("This is not evidence of growth over time; multiple survey years would be needed to measure that.")


## 8. Conclusions — write after running the notebook

After running all cells, record:
- Top current programming languages.
- Top wanted programming languages.
- Countries with the most survey responses.
- Whether median compensation generally changes across experience bands.
- How employment shares differ by education level.
- Technologies with the largest positive interest gap.

Avoid causal claims. This survey is observational and represents respondents, not every developer worldwide.